# NumCompute: Streaming Data Processing and Incremental Learning

This notebook demonstrates the streaming functionality developed for Assignment 2.2.

It demonstrates how to:
- Load numeric CSV data using the custom NumCompute I/O function.
- Process data in smaller chunks.
- Train pipelines incrementally using `partial_fit()`.
- Record accuracy and error as new chunks arrive.
- Visualise performance and compare two decision-tree models.

A small synthetic CSV file is created for this demonstration. It is sample data, not an external dataset.

## 1. Import Required Modules

Import NumPy, Matplotlib, and the custom NumCompute modules.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "demo" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from numcompute.io import load_csv_chunks
from numcompute_stream.tree import DecisionTreeClassifier
from numcompute_stream.preprocessing import StandardScaler
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.visualise import (
    plot_metric_over_time,
    compare_models,
    plot_predictions_vs_ground_truth
)

print("Imports completed successfully!")

## 2. Create a Sample CSV File

Create a small numeric classification CSV with two input features and one target column. This provides data for demonstrating the CSV loader without relying on an external dataset.

In [ ]:
data = np.array([
    [1.0, 1.2, 0], [1.3, 1.8, 0], [1.7, 1.4, 0], [2.0, 2.1, 0],
    [2.2, 1.7, 0], [2.5, 2.4, 0], [2.8, 2.0, 0], [3.0, 2.7, 0],
    [5.0, 5.2, 1], [5.3, 5.8, 1], [5.7, 5.4, 1], [6.0, 6.1, 1],
    [6.2, 5.7, 1], [6.5, 6.4, 1], [6.8, 6.0, 1], [7.0, 6.7, 1],
    [1.1, 1.5, 0], [1.6, 2.0, 0], [2.1, 1.9, 0], [2.6, 2.3, 0],
    [5.1, 5.5, 1], [5.6, 6.0, 1], [6.1, 5.8, 1], [6.6, 6.3, 1]
])

csv_path = Path("stream_sample.csv")
np.savetxt(csv_path, data, delimiter=",", fmt="%.2f")
print("Sample CSV created:", csv_path)
print("Dataset shape:", data.shape)

## 3. Load the CSV File in Chunks

Use `load_csv_chunks()` from `numcompute.io` to read the CSV in batches, simulating data arriving over time.

In [ ]:
chunk_size = 4
data_chunks = list(load_csv_chunks(filepath=str(csv_path), chunk_size=chunk_size))

print("Number of chunks:", len(data_chunks))
for i, chunk in enumerate(data_chunks):
    print(f"Chunk {i + 1}: {chunk.shape}")

## 4. Separate Input Features and Target Labels

The first two columns are input features (`X`). The last column contains the target label (`y`).

In [ ]:
X_chunks = [chunk[:, :-1] for chunk in data_chunks]
y_chunks = [chunk[:, -1].astype(int) for chunk in data_chunks]

for i in range(len(X_chunks)):
    print(f"Chunk {i + 1}: features={X_chunks[i].shape}, labels={y_chunks[i].shape}")

## 5. Create Streaming Pipelines

Create two pipelines for comparison. Each applies `StandardScaler` before a decision tree. The trees use different maximum depths and a buffer size of four rows.

In [ ]:
pipeline_1 = Pipeline([
    ("scaler", StandardScaler()),
    ("tree", DecisionTreeClassifier(max_depth=2, buffer_size=chunk_size, random_state=42))
])

pipeline_2 = Pipeline([
    ("scaler", StandardScaler()),
    ("tree", DecisionTreeClassifier(max_depth=3, buffer_size=chunk_size, random_state=42))
])

print("Both streaming pipelines created successfully!")

## 6. Incremental Training and Evaluation

The first chunk is used for initial training. Each later chunk is first used to evaluate the current model. Accuracy and error are recorded, and then the model is updated using `partial_fit()`.

This is a simple prequential evaluation approach: each chunk is evaluated before it is used for the next model update.

In [ ]:
accuracy_values = []
error_values = []
comparison_accuracy = []
comparison_error = []
chunk_numbers = []

last_y_true = None
last_y_pred = None

# Initial training on the first chunk
pipeline_1.partial_fit(X_chunks[0], y_chunks[0])
pipeline_2.partial_fit(X_chunks[0], y_chunks[0])
print("Chunk 1 used for initial training.")

# Evaluate each later chunk, then update both pipelines
for i in range(1, len(X_chunks)):
    y_pred_1 = pipeline_1.predict(X_chunks[i])
    y_pred_2 = pipeline_2.predict(X_chunks[i])

    accuracy_1 = np.mean(y_pred_1 == y_chunks[i])
    error_1 = 1.0 - accuracy_1
    accuracy_2 = np.mean(y_pred_2 == y_chunks[i])
    error_2 = 1.0 - accuracy_2

    accuracy_values.append(accuracy_1)
    error_values.append(error_1)
    comparison_accuracy.append(accuracy_2)
    comparison_error.append(error_2)
    chunk_numbers.append(i + 1)

    last_y_true = y_chunks[i]
    last_y_pred = y_pred_1

    print(f"Chunk {i + 1}:")
    print(f"  Tree depth 2 - accuracy: {accuracy_1:.2f}, error: {error_1:.2f}")
    print(f"  Tree depth 3 - accuracy: {accuracy_2:.2f}, error: {error_2:.2f}")

    # Update the models after evaluation
    pipeline_1.partial_fit(X_chunks[i], y_chunks[i])
    pipeline_2.partial_fit(X_chunks[i], y_chunks[i])

print("Incremental training and evaluation completed!")

## 7. Visualise Accuracy Over Time

Use the custom `plot_metric_over_time()` function to show how accuracy changes across incoming chunks.

In [ ]:
plot_metric_over_time(
    accuracy_values,
    title="Streaming Model Accuracy",
    ylabel="Accuracy"
)

## 8. Visualise Error Over Time

Plot the classification error recorded for each incoming chunk.

In [ ]:
plot_metric_over_time(
    error_values,
    title="Streaming Model Error",
    ylabel="Error"
)

## 9. Compare the Two Models

Use the custom `compare_models()` function to compare the accuracy of the two decision-tree pipelines.

In [ ]:
compare_models(
    accuracy_values,
    comparison_accuracy,
    labels=("Tree Depth 2", "Tree Depth 3")
)

## 10. Compare Predictions with Ground Truth

Compare predictions from the first pipeline with actual labels from the final evaluated chunk.

In [ ]:
plot_predictions_vs_ground_truth(last_y_true, last_y_pred)

## 11. Summary of Results

Print the accuracy and error recorded for each evaluated chunk. The first chunk is excluded because it was used for initial training.

In [ ]:
for i in range(len(chunk_numbers)):
    print(f"Chunk {chunk_numbers[i]}:")
    print(f"  Tree depth 2 - accuracy: {accuracy_values[i]:.2f}, error: {error_values[i]:.2f}")
    print(f"  Tree depth 3 - accuracy: {comparison_accuracy[i]:.2f}, error: {comparison_error[i]:.2f}")

print("\nDemo completed successfully!")